# AlpCAN — LIDC-IDRI Veri Keşfi

**Amaç:** LIDC-IDRI veri setini keşfet, nodül dağılımlarını analiz et, nnU-Net eğitimi için veri hazırlığını yap.

**Veri Seti:** LIDC-IDRI (Lung Image Database Consortium) — 1.018 BT tarama, 7.371 radyolog tarafından etiketlenmiş lezyon.

**Kaggle Dataset:** `andyczhao/lidc-idri-lung-nodule-dataset`

---
**Kaggle Ayarları:**
1. Bu notebook'u Kaggle'a yükle
2. Sağ panelden **Add Data** → `andyczhao/lidc-idri-lung-nodule-dataset` ekle
3. **Accelerator** → GPU T4 x2 seç
4. **Internet** → On yap (pip install için)

In [ ]:
# Gerekli kütüphaneler
!pip install -q pylidc SimpleITK nibabel pydicom matplotlib seaborn pandas numpy

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Kaggle veri yolu
DATA_DIR = Path("/kaggle/input/lidc-idri-lung-nodule-dataset")

# Veri setinin yapısını kontrol et
print("Veri seti dizini:")
for item in sorted(DATA_DIR.iterdir()):
    if item.is_dir():
        sub_count = len(list(item.iterdir()))
        print(f"  {item.name}/ ({sub_count} öğe)")
    else:
        size_mb = item.stat().st_size / (1024*1024)
        print(f"  {item.name} ({size_mb:.1f} MB)")

## 1. DICOM Dosya Yapısını Keşfet

In [ ]:
import pydicom

# İlk DICOM dosyasını bul ve oku
dcm_files = sorted(glob.glob(str(DATA_DIR / "**/*.dcm"), recursive=True))
print(f"Toplam DICOM dosyası: {len(dcm_files):,}")

if dcm_files:
    # İlk dosyayı incele
    ds = pydicom.dcmread(dcm_files[0])
    print(f"\nÖrnek DICOM header:")
    print(f"  Patient ID: {ds.PatientID if hasattr(ds, 'PatientID') else 'N/A'}")
    print(f"  Modality: {ds.Modality if hasattr(ds, 'Modality') else 'N/A'}")
    print(f"  Rows x Cols: {ds.Rows}x{ds.Columns}")
    print(f"  Slice Thickness: {ds.SliceThickness if hasattr(ds, 'SliceThickness') else 'N/A'} mm")
    print(f"  Pixel Spacing: {ds.PixelSpacing if hasattr(ds, 'PixelSpacing') else 'N/A'}")
    print(f"  Bits Allocated: {ds.BitsAllocated}")

## 2. Hasta ve Tarama İstatistikleri

In [ ]:
# Hasta dizinlerini say
patient_dirs = sorted([d for d in DATA_DIR.rglob("LIDC-IDRI-*") if d.is_dir()])
print(f"Toplam hasta dizini: {len(patient_dirs)}")

# Her hastanın DICOM sayısını hesapla
patient_stats = []
for pdir in patient_dirs[:50]:  # İlk 50 hasta (hız için)
    dcms = list(pdir.rglob("*.dcm"))
    patient_stats.append({
        "patient_id": pdir.name,
        "slice_count": len(dcms),
    })

df_patients = pd.DataFrame(patient_stats)
print(f"\nDilim sayısı istatistikleri (ilk 50 hasta):")
print(df_patients["slice_count"].describe())

## 3. Nodül Etiketlerini Yükle (pylidc)

In [ ]:
# pylidc konfigürasyonu
# Not: pylidc LIDC-IDRI'nin belirli bir dizin yapısını bekler
# Kaggle'daki veri seti yapısına göre ayarlanmalı

try:
    import pylidc as pl
    
    # pylidc config dosyası oluştur
    pylidc_conf = os.path.expanduser("~/.pylidcrc")
    with open(pylidc_conf, "w") as f:
        f.write(f"[dicom]\npath = {DATA_DIR}\n")
    
    scans = pl.query(pl.Scan).all()
    print(f"pylidc ile bulunan tarama sayısı: {len(scans)}")
    
except Exception as e:
    print(f"pylidc yüklenemedi veya veri formatı uyumsuz: {e}")
    print("Manuel etiket analizi yapılacak...")

In [ ]:
# Alternatif: CSV/XML etiket dosyalarını ara
annotation_files = list(DATA_DIR.rglob("*.xml")) + list(DATA_DIR.rglob("*.csv"))
print(f"Bulunan etiket dosyaları: {len(annotation_files)}")
for f in annotation_files[:10]:
    print(f"  {f.relative_to(DATA_DIR)}")

## 4. Örnek BT Görüntüleme

In [ ]:
import SimpleITK as sitk

def load_ct_scan(patient_dir):
    """Bir hastanın DICOM serisini yükle."""
    reader = sitk.ImageSeriesReader()
    dicom_files = reader.GetGDCMSeriesFileNames(str(patient_dir))
    if not dicom_files:
        # Alt dizinlerde ara
        for subdir in patient_dir.rglob("*"):
            if subdir.is_dir():
                dicom_files = reader.GetGDCMSeriesFileNames(str(subdir))
                if dicom_files:
                    break
    
    if not dicom_files:
        return None
    
    reader.SetFileNames(dicom_files)
    image = reader.Execute()
    return image

# İlk hastayı yükle
if patient_dirs:
    print(f"Yükleniyor: {patient_dirs[0].name}")
    ct_image = load_ct_scan(patient_dirs[0])
    if ct_image:
        ct_array = sitk.GetArrayFromImage(ct_image)
        spacing = ct_image.GetSpacing()
        print(f"  Boyut: {ct_array.shape}")
        print(f"  Spacing: {spacing}")
        print(f"  HU aralığı: [{ct_array.min()}, {ct_array.max()}]")
        print(f"  Dtype: {ct_array.dtype}")
    else:
        print("  DICOM serisi bulunamadı")

In [ ]:
# BT kesitlerini görselleştir
if ct_image:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"{patient_dirs[0].name} — BT Kesitleri", fontsize=14)
    
    n_slices = ct_array.shape[0]
    slice_indices = np.linspace(n_slices * 0.2, n_slices * 0.8, 8, dtype=int)
    
    for idx, ax in zip(slice_indices, axes.flat):
        ax.imshow(ct_array[idx], cmap="gray", vmin=-1000, vmax=400)
        ax.set_title(f"Dilim {idx}/{n_slices}", fontsize=10)
        ax.axis("off")
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/ct_slices_preview.png", dpi=150)
    plt.show()
    print("Kaydedildi: ct_slices_preview.png")

## 5. HU Normalizasyon ve Akciğer Maskesi (Ajan 1 Prototipi)

In [ ]:
from scipy import ndimage

def apply_hu_clipping(volume, hu_min=-1000, hu_max=400):
    """HU değerlerini kırp — akciğer dokusu optimize aralığı."""
    return np.clip(volume, hu_min, hu_max)

def create_lung_mask(volume, threshold=-600):
    """Eşik tabanlı akciğer maskesi oluştur."""
    # 1. Eşikleme
    binary = volume < threshold
    
    # 2. Bağlı bileşen analizi
    labeled, num_features = ndimage.label(binary)
    
    # 3. En büyük 2 bileşeni bul (sol ve sağ akciğer)
    component_sizes = ndimage.sum(binary, labeled, range(1, num_features + 1))
    
    if len(component_sizes) >= 2:
        largest_two = np.argsort(component_sizes)[-2:] + 1
        mask = np.isin(labeled, largest_two)
    elif len(component_sizes) == 1:
        mask = labeled == 1
    else:
        mask = binary
    
    # 4. Morfolojik kapanma (delikleri doldur)
    mask = ndimage.binary_closing(mask, iterations=3)
    mask = ndimage.binary_fill_holes(mask)
    
    return mask.astype(np.uint8)

if ct_image:
    # HU kırpma
    ct_clipped = apply_hu_clipping(ct_array)
    
    # Akciğer maskesi — orta dilimde test et
    mid_slice = ct_array.shape[0] // 2
    mask_2d = create_lung_mask(ct_array[mid_slice])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(ct_array[mid_slice], cmap="gray", vmin=-1000, vmax=400)
    axes[0].set_title("Orijinal BT")
    axes[1].imshow(mask_2d, cmap="gray")
    axes[1].set_title("Akciğer Maskesi")
    axes[2].imshow(ct_array[mid_slice] * mask_2d, cmap="gray", vmin=-1000, vmax=400)
    axes[2].set_title("Maskelenmiş BT")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/lung_mask_preview.png", dpi=150)
    plt.show()
    print("Ajan 1 prototipi çalışıyor!")

## 6. Veri Seti Özet Raporu

In [ ]:
# Tüm hastaların temel istatistiklerini topla
print("="*60)
print("LIDC-IDRI VERİ SETİ ÖZET RAPORU")
print("="*60)
print(f"Toplam hasta dizini: {len(patient_dirs)}")
print(f"Toplam DICOM dosyası: {len(dcm_files):,}")
if ct_image:
    print(f"\nÖrnek tarama boyutu: {ct_array.shape}")
    print(f"Örnek spacing: {spacing}")
    print(f"HU aralığı: [{ct_array.min()}, {ct_array.max()}]")
print(f"\nBulunan etiket dosyaları: {len(annotation_files)}")
print("\n" + "="*60)
print("SONRAKİ ADIM: 02_nnunet_data_preparation.ipynb")
print("  → LIDC-IDRI'yi nnU-Net formatına dönüştür")
print("  → Nodül segmentasyon maskeleri oluştur")
print("  → 5-fold cross-validation split")
print("="*60)